# Cancer Mutation Detection - Pipeline Complet

**Prédiction de Mutations Génétiques Pathogènes au Cancer**

Ce notebook exécute l'ensemble du pipeline ML:
1. **Chargement** des données ClinVar
2. **Prétraitement** et nettoyage
3. **Entraînement** des 3 modèles (XGBoost, NN Simple, NN Avancé)
4. **Évaluation** et comparaison
5. **Prédiction** sur cas réels

In [1]:
# Configuration des imports
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Fixer le seed pour reproductibilité
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✓ Imports configurés")
print(f"✓ Seed: {RANDOM_STATE}")

✓ Imports configurés
✓ Seed: 42


---

## ÉTAPE 1: Chargement des Données

In [2]:
from data_loader import load_clinvar_data, explore_data, get_features_info

# Charger le dataset
# Utiliser nrows=50000 pour un test rapide, ou None pour tout charger
df = load_clinvar_data('../data/clinvar_conflicting.csv', nrows=None)

# Exploration rapide
stats = explore_data(df)

# Afficher les features sélectionnées
features_info = get_features_info()
print("\n" + "="*60)
print("FEATURES SÉLECTIONNÉES")
print("="*60)
for feat, info in features_info.items():
    print(f"\n{feat}:")
    print(f"  Description: {info['description']}")
    print(f"  Importance: {info['importance']}")

Chargement des données depuis: ../data/clinvar_conflicting.csv
Dataset chargé: 65,188 lignes, 46 colonnes

EXPLORATION DES DONNÉES

Dimensions: 65,188 lignes × 46 colonnes
Mémoire utilisée: 125.27 MB

Distribution de la cible (CLNSIGINCL):
CLNSIGINCL
NaN                                    65021
495755:Pathogenic                          2
2067:risk_factor                           2
13906:Pathogenic                           2
446729:other                               2
                                       ...  
3863:Pathogenic                            1
424765:Pathogenic|437912:Pathogenic        1
424775:Likely_pathogenic                   1
217885:Pathogenic                          1
10361:Pathogenic|10382:other               1
Name: count, Length: 138, dtype: int64

FEATURES SÉLECTIONNÉES

CADD_PHRED:
  Description: Score de pathogénicité combiné (0-100)
  Importance: Très élevée - Score expert #1

CADD_RAW:
  Description: Version brute du score CADD
  Importance: Élevée - Com

---

## ÉTAPE 2: Prétraitement

In [6]:
import sys
sys.path.insert(0, '../src') 

from preprocessing import clean_clinvar_data, prepare_train_test_data

# Nettoyer les données
df_clean, features = clean_clinvar_data(df)

# Préparer train/test
X_train, X_test, y_train, y_test, scaler, feature_names = prepare_train_test_data(
    df_clean, 
    test_size=0.2, 
    random_state=RANDOM_STATE
)

print(f"\n✓ Prétraitement terminé")
print(f"  Features: {feature_names}")


NETTOYAGE DES DONNÉES
Données brutes: 65,188 lignes

Distribution de la cible (CLASS):
  Benign (0):     48,754 (74.8%)
  Pathogenic (1): 16,434 (25.2%)

SIFT encodé: {0.0: 12561, 1.0: 12275}
PolyPhen encodé: {0.0: 13329, 1.0: 7531, 0.5: 3932}
  CADD_PHRED: 1,092 valeurs manquantes → médiane (14.0900)
  CADD_RAW: 1,092 valeurs manquantes → médiane (1.6429)
  SIFT: 40,352 valeurs manquantes → médiane (0.0000)
  PolyPhen: 40,396 valeurs manquantes → médiane (0.0000)
  BLOSUM62: 39,595 valeurs manquantes → médiane (-1.0000)

Dataset final: 65,188 lignes × 8 colonnes
Features utilisées: ['CADD_PHRED', 'CADD_RAW', 'SIFT', 'PolyPhen', 'AF_EXAC', 'AF_TGP', 'BLOSUM62']

PRÉPARATION TRAIN/TEST
Features shape: (65188, 7)
Target shape: (65188,)

Train set: 52,150 samples
  Benign: 39,003 (74.8%)
  Pathogenic: 13,147 (25.2%)

Test set: 13,038 samples
  Benign: 9,751 (74.8%)
  Pathogenic: 3,287 (25.2%)

Normalisation effectuée (StandardScaler)

✓ Prétraitement terminé
  Features: ['CADD_PHRED', 'C

---

## ÉTAPE 3: Entraînement des Modèles

In [9]:
from train import train_all_models

# Paramètres d'entraînement
xgb_params = {}
nn_simple_params = {'epochs': 50, 'batch_size': 32, 'validation_split': 0.2}
nn_advanced_params = {'epochs': 100, 'batch_size': 32, 'validation_split': 0.2}

# Entraîner tous les modèles
training_results = train_all_models(
    X_train, y_train,
    xgb_params=xgb_params,
    nn_simple_params=nn_simple_params,
    nn_advanced_params=nn_advanced_params,
    save=True,
    output_dir='../models',
    random_state=RANDOM_STATE
)

# Extraire les modèles
xgb_model = training_results['xgboost']['model']
nn_simple = training_results['nn_simple']['model']
nn_advanced = training_results['nn_advanced']['model']
nn_simple_history = training_results['nn_simple']['history']
nn_advanced_history = training_results['nn_advanced']['history']


ENTRAÎNEMENT XGBOOST
Distribution des classes: 39003 négatifs, 13147 positifs
scale_pos_weight: 2.97
Paramètres: n_estimators=200, max_depth=8, learning_rate=0.05
Samples d'entraînement: 52,150
✓ Entraînement terminé

ENTRAÎNEMENT NEURAL NETWORK SIMPLE
Architecture: 128 → 64 → 32 → 1
Epochs: 50, Batch size: 32
Validation split: 0.2
Class weights: {0: 0.6685383175653155, 1: 1.9833422073476838}


2026-04-14 21:54:11.116035: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2026-04-14 21:54:11.116254: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2026-04-14 21:54:11.116297: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcublas.so.11'; dlerror: libcublas.so.11: cannot open shared object file: No such file or directory
2026-04-14 21:54:11.116325: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcublasLt.so.11'; dlerror: libcublasLt.so.11: cannot open shared object file: No such file or directory
2026-04-14 21:54:11.116351: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Co

Epoch 1/50
1304/1304 - 7s - loss: 0.6984 - auc: 0.6027 - precision: 0.2967 - recall: 0.7837 - val_loss: 0.6879 - val_auc: 0.6167 - val_precision: 0.2937 - val_recall: 0.8955 - lr: 0.0010 - 7s/epoch - 5ms/step
Epoch 2/50
1304/1304 - 2s - loss: 0.6646 - auc: 0.6123 - precision: 0.2977 - recall: 0.8612 - val_loss: 0.6493 - val_auc: 0.6149 - val_precision: 0.3002 - val_recall: 0.8844 - lr: 0.0010 - 2s/epoch - 1ms/step
Epoch 3/50
1304/1304 - 2s - loss: 0.6602 - auc: 0.6117 - precision: 0.2973 - recall: 0.8767 - val_loss: 0.6694 - val_auc: 0.6157 - val_precision: 0.2940 - val_recall: 0.9131 - lr: 0.0010 - 2s/epoch - 1ms/step
Epoch 4/50
1304/1304 - 2s - loss: 0.6581 - auc: 0.6121 - precision: 0.2981 - recall: 0.8813 - val_loss: 0.6368 - val_auc: 0.6146 - val_precision: 0.2974 - val_recall: 0.8473 - lr: 0.0010 - 2s/epoch - 1ms/step
Epoch 5/50
1304/1304 - 2s - loss: 0.6566 - auc: 0.6154 - precision: 0.2970 - recall: 0.8856 - val_loss: 0.6641 - val_auc: 0.6179 - val_precision: 0.2946 - val_recal

TypeError: unsupported operand type(s) for *: 'ExponentialDecay' and 'int'

---

## ÉTAPE 4: Évaluation

In [ ]:
from evaluate import evaluate_all_models

# Évaluer tous les modèles et générer les visualisations
eval_results = evaluate_all_models(
    xgb_model, nn_simple, nn_advanced,
    X_test, y_test, feature_names,
    nn_simple_history.history if nn_simple_history else None,
    nn_advanced_history.history if nn_advanced_history else None,
    output_dir='../results'
)

---

## ÉTAPE 5: Prédiction sur Cas Réels

In [ ]:
# Exemple de mutation pathogène (Cancer)
example_pathogenic = np.array([[
    28.0,      # CADD_PHRED (élevé = pathogène)
    3.5,       # CADD_RAW
    0.01,      # SIFT (bas = délétère)
    0.95,      # PolyPhen (élevé = dangereux)
    0.0001,    # AF_EXAC (rare)
    0.0002,    # AF_TGP (rare)
    -3.0       # BLOSUM62 (négatif = grave)
]])

# Exemple de mutation bénigne
example_benign = np.array([[
    5.0,       # CADD_PHRED (faible)
    0.5,       # CADD_RAW
    0.8,       # SIFT (élevé = toléré)
    0.1,       # PolyPhen (faible)
    0.35,      # AF_EXAC (commun)
    0.40,      # AF_TGP (commun)
    2.0        # BLOSUM62 (positif)
]])

# Normaliser
example_pathogenic_scaled = scaler.transform(example_pathogenic)
example_benign_scaled = scaler.transform(example_benign)

for name, example, example_scaled in [
    ("MUTATION PATHOGÈNE (Cancer)", example_pathogenic, example_pathogenic_scaled),
    ("MUTATION BÉNIGNE (Inoffensive)", example_benign, example_benign_scaled)
]:
    print(f"\n{name}:")
    print(f"  Features: CADD_PHRED={example[0][0]:.1f}, SIFT={example[0][2]:.2f}, PolyPhen={example[0][3]:.2f}")
    
    pred_xgb = xgb_model.predict_proba(example_scaled)[0, 1]
    pred_nn_simple = nn_simple.predict(example_scaled, verbose=0)[0, 0]
    pred_nn_advanced = nn_advanced.predict(example_scaled, verbose=0)[0, 0]
    pred_avg = (pred_xgb + pred_nn_simple + pred_nn_advanced) / 3
    
    print(f"  Probabilité Pathogène:")
    print(f"    XGBoost:    {pred_xgb:.1%}")
    print(f"    NN Simple:  {pred_nn_simple:.1%}")
    print(f"    NN Avancé:  {pred_nn_advanced:.1%}")
    print(f"    Moyenne:    {pred_avg:.1%}")
    
    verdict = "PATHOGÈNE ⚠️" if pred_avg > 0.5 else "BÉNIGNE ✓"
    print(f"  → VERDICT: {verdict}")

---

## Résumé Final

In [ ]:
# Trouver le meilleur modèle
roc_aucs = [r['roc_auc'] for r in eval_results]
best_idx = np.argmax(roc_aucs)
best_model = eval_results[best_idx]

print("\n" + "="*60)
print("RÉSUMÉ FINAL")
print("="*60)
print(f"\n🏆 Meilleur modèle: {best_model['model_name']}")
print(f"   ROC-AUC: {best_model['roc_auc']:.4f}")

if best_model['roc_auc'] >= 0.80:
    print(f"\n✅ OBJECTIF ATTEINT: ROC-AUC ≥ 80% ({best_model['roc_auc']:.1%})")
else:
    print(f"\n⚠️  Objectif non atteint: ROC-AUC < 80% ({best_model['roc_auc']:.1%})")

print(f"\n📁 Résultats sauvegardés dans: ../results/")
print("="*60)

---

## Visualisations Interactives

In [ ]:
# Afficher les images générées
from IPython.display import Image, display

print("\n📊 Courbes ROC:")
display(Image(filename='../results/roc_curves.png'))

print("\n📊 Matrices de Confusion:")
display(Image(filename='../results/confusion_matrices.png'))

print("\n📊 Feature Importance:")
display(Image(filename='../results/feature_importance.png'))

print("\n📊 Historique d'entraînement:")
display(Image(filename='../results/training_history.png'))

---

## Tableau Comparatif

In [ ]:
# Afficher le tableau comparatif
comparison_df = pd.read_csv('../results/model_comparison.csv')
comparison_df